In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from tavily import TavilyClient
from langgraph.config import get_stream_writer
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage

GEMINI_MODEL = 'gemini-2.5-flash'
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')

model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    api_key=GEMINI_API_KEY,
    temperature=0.3, 
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [20]:
from typing import TypedDict, Annotated
from uuid import uuid4
#Questa funzione mi serve per sostituire i messaggi all'interno dello stato

def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    """Se il messaggio non ha un id lo setto e poi se in ingresso ho un messaggio con
    un id già presente nella lista messaggi lo sostituisco"""
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]

In [31]:
class Agent:

    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_gemini)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer = checkpointer, interrupt_before=['action'])
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_gemini(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        writer = get_stream_writer()
        for t in tool_calls:
            writer(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
                writer(result)

            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        return {'messages': results}

In [14]:
from langchain.tools import tool

@tool
def tavily_search_tool(
    query: str, max_results: int = 5
) -> list[dict]:
    """
    Perform a search using the Tavily API.

    Args:
        query (str): The search query.
        max_results (int): Number of results to return (default 5).
        include_images (bool): Whether to include image results.

    Returns:
        List[dict]: A list of dictionaries with keys like 'title', 'content', and 'url'.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables.")

    client = TavilyClient(api_key)

    try:
        response = client.search(
            query=query, max_results=max_results
        )

        results = []
        for r in response.get("results", []):
            results.append(
                {
                    "title": r.get("title", ""),
                    "content": r.get("content", ""),
                    "url": r.get("url", ""),
                }
            )

        return results

    except Exception as e:
        return [{"error": str(e)}]  # For LLM-friendly agents

In [15]:
from langgraph.checkpoint.redis import RedisSaver
from redis import Redis
DB_URI = "redis://localhost:6379"
with RedisSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup()

In [36]:
agent = Agent(model=model, system = 'Sei un assistente AI che ha a disposizione dei tool per cercare informazioni online per rispondere alle richieste dell utente. La data corrente è 16/04/2026', tools = [tavily_search_tool], checkpointer = checkpointer)

In [37]:
config = {
    "configurable": {
        "thread_id": "01234"
    }
}
for event_type, event in agent.graph.stream({"messages": [HumanMessage('Chi ha vinto il campionato di calcio serie A nel 2025? Qual è il giocatore più pagato quell anno di quella squadra?')]}, config = config, stream_mode=['updates']):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"query": "chi ha vinto il campionato di calcio serie A nel 2025"}'}, '__gemini_function_call_thought_signatures__': {'292c114e-76e7-4025-88cc-886e22e8c570': 'CsoDAb4+9vtCpULyx9fCiWwtNczV5pXGkfHq4cN+/kIo6rkWj7oVFPnQ+uXsWsjOOXVKspr4Wi6kRb4dhrvtW8hgpEDamFfPefAqrQ3qqg2q4l3BTSX6c4/d0fLxkXxYh7d2cOCosnc/NqNJiUojFyhBcWpHC0lYIyQqHkPAokU1oXPq1850DH+JkcGb5cjB0X8RL8CzxcaKn/ziaDFIn+Xkj9/MR5Q9/SmRoGgmlfgwQGcAY9CltIsl5DByxyLGcaxNEFifX3G788yV0O1WpIPTkKzdZ/yQDCMbHGXsv7J+KTONxlpXd3y5eBa4+KkRGOj8LeYrEoMvf6p7g9fdyO5M6hZilnx8jJxI0zH/TdNlOHttB95XUlin1g9y+zeGSSiaET+zwAIZVIS5Kxwy98LQWAMzVKkb2CpEFLMtjNgVRU3VyJ8I2oZRzvK4pnBNT5TUpbK1d6MCR8GzZUqWPHtdo6Ni4SB1e8VNN9HmngFjDLFWlDvEmsidbDKKzyMle9hnTO5dzcFCACp/p/0y4CNu0h+MUSCJf57iO5Qsz8C/Hj4BeHtwgbUMlJ/MJ7P37rTsEJqUU1cjbo4364uxbaxEi6yIMoLwKtZOQBE='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'mo

lo streaming si è fermato prima del nodo action

In [38]:
agent.graph.get_state(config) # restituisce uno snapshot dello stato

StateSnapshot(values={'messages': [HumanMessage(content='Chi ha vinto il campionato di calcio serie A nel 2025? Qual è il giocatore più pagato quell anno di quella squadra?', additional_kwargs={}, response_metadata={}, id='9e1937c4-55be-40aa-ae64-d59007a885fb'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"query": "chi ha vinto il campionato di calcio serie A nel 2025"}'}, '__gemini_function_call_thought_signatures__': {'292c114e-76e7-4025-88cc-886e22e8c570': 'CsoDAb4+9vtCpULyx9fCiWwtNczV5pXGkfHq4cN+/kIo6rkWj7oVFPnQ+uXsWsjOOXVKspr4Wi6kRb4dhrvtW8hgpEDamFfPefAqrQ3qqg2q4l3BTSX6c4/d0fLxkXxYh7d2cOCosnc/NqNJiUojFyhBcWpHC0lYIyQqHkPAokU1oXPq1850DH+JkcGb5cjB0X8RL8CzxcaKn/ziaDFIn+Xkj9/MR5Q9/SmRoGgmlfgwQGcAY9CltIsl5DByxyLGcaxNEFifX3G788yV0O1WpIPTkKzdZ/yQDCMbHGXsv7J+KTONxlpXd3y5eBa4+KkRGOj8LeYrEoMvf6p7g9fdyO5M6hZilnx8jJxI0zH/TdNlOHttB95XUlin1g9y+zeGSSiaET+zwAIZVIS5Kxwy98LQWAMzVKkb2CpEFLMtjNgVRU3VyJ8I2oZRzvK4pnBNT5TUpbK1d6MCR8GzZUqWPHtdo6N

In [42]:
#Qual'è il successivo nodo del grafo?
agent.graph.get_state(config).next

('action',)

In [45]:
#Continuiamo con l'esecuzione
for event in agent.graph.stream(None, config):
    for v in event.values():
        print(v)

{'messages': [ToolMessage(content='[{\'title\': \'Calcio Serie A - Classifica Finale 2024/2025 Napoli Campione - Italiavista.it\', \'content\': \'Il campionato, 123ª edizione della massima serie italiana e 93ª a girone unico, ha visto l’Inter chiudere al secondo posto con un solo punto di distacco, mentre l’Atalanta ha conquistato il podio sorprendendo molti pronostici iniziali. La Serie A rappresenta il vertice del calcio professionistico italiano, organizzata in regime di girone unico con partite di andata e ritorno per un totale di 38 giornate. Con le tre retrocesse che salutano la massima serie, il campionato si prepara ora alla nuova edizione 2025/2026, già iniziata con nuovi equilibri e nuove sfide. Il campionato italiano di massima serie vede la partecipazione di 20 squadre che si affrontano in un girone unico con formula di andata e ritorno, per un totale di 38 giornate. Sì, oltre alla sospensione di Fiorentina-Inter per il malore di un giocatore, si segnalano i rinvii della 33

In [46]:
agent.graph.get_state(config).next

('action',)

In [47]:
#Continuiamo con l'esecuzione
for event in agent.graph.stream(None, config):
    for v in event.values():
        print(v)

{'messages': [ToolMessage(content='[{\'title\': \'Monte ingaggi Napoli 2025/26: tutti gli stipendi aggiornati degli azzurri | Transfermarkt\', \'content\': "Kevin De Bruyne resta il giocatore più pagato della rosa con 11,11 milioni di euro lordi, davanti a Rasmus Højlund e Romelu Lukaku. Gli stipendi più alti della Serie A  La Top11 per stipendi del Napoli 2025/26 totalizza 80 milioni di euro lordi. Ieri 10:51 Serie A Media spettatori: Inter e Milan guadagnano un posto, ma chi è davanti? Ieri 10:51 Transfermarkt Media spettatori: Inter e Milan guadagnano un posto, ma chi è davanti? 13/04/2026 14:45 Transfermarkt Serie A e giovani: Como, valori alle stelle ed esempio per le big. 10/04/2026 14:45 Transfermarkt Calendario: ecco il (lieve) vantaggio del Napoli. 08/02/2026 07:45 Transfermarkt Stipendi Top in Serie A: il rinnovo di Yildiz sposta gli equilibri? 01/12/2025 07:46 Transfermarkt Quanto vale il Napoli? 24/03/2026 13:00 Transfermarkt Serie A coi nuovi valori: crolla l\'Inter ma sal

In [53]:
# Inseriamo l'interazione umana
config = config = {
    "configurable": {
        "thread_id": "012345678"
    }
}
for event in agent.graph.stream({"messages": [HumanMessage('Chi ha vinto il campionato di calcio serie A nel 2025? Qual è il giocatore più pagato quell anno di quella squadra?')]}, config):
    for v in event.values():
        print(v)
        
while agent.graph.get_state(config).next:
    print("\n", agent.graph.get_state(config),"\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in agent.graph.stream(None, config):
        for v in event.values():
            print(v)

{'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"query": "chi ha vinto campionato serie A 2025"}'}, '__gemini_function_call_thought_signatures__': {'e854dbad-7235-48c4-bc9e-e361d45d48d7': 'CsIDAb4+9vu7oMEHj2Pl4xaan9J4fjlb4qWpJbwg/pJCIcq9Ay0jrNtwzwvvw1a18Y+7giAP80KKty8/cMpF3oYr7qqne+Dx+c52YDDZ4J0k+g8eJsQRYMHsXC3gg9kTPrGUkF/VV1tVdu1/hTAZJ51L9obDVBpGwov3e1QSPvnxZhCN15P+Ll8t1DXanSTkQcdtbQ5dyfUMWVvQpOGU4vQnRdykZJ4br4tGC9QARkhpveFriQucMxz9Imobo9NIuvgPBCQfbX0uwQlLlrzCCxcDzHAM+YULeY36w23r4admufTiTA5iwSB6uGGuTcOo0x9VWM5HimFGfHmDCqkEl9w/MnL8s/p85zwllC53r+yznP1yc4sks+bxdsbIb60qSMIA1uIEZ+yR+8FPUCcLnygZK2Lv02HCuyG3JkKkzurd8dxTnbYUleQLZy3rEZ3+Oosoz70hN+2oq5faakm3Jnj/2iPFfi3HNT+Q6iBbGXDClbuTOgkZmpCFgQQrgXw0mpprO+nreDUG2Fc+rbmRNjlSXhV9D3GVHGY0XECygDaWNo8pdwa5Qxee3Ez+xDBEbSzgEtu8S03i71fuCUDiJ1udYhem'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'

proceed? y


{'messages': [ToolMessage(content='[{\'title\': \'Calcio Serie A - Classifica Finale 2024/2025 Napoli Campione\', \'content\': \'Il campionato, 123ª edizione della massima serie italiana e 93ª a girone unico, ha visto l’Inter chiudere al secondo posto con un solo punto di distacco, mentre l’Atalanta ha conquistato il podio sorprendendo molti pronostici iniziali. La Serie A rappresenta il vertice del calcio professionistico italiano, organizzata in regime di girone unico con partite di andata e ritorno per un totale di 38 giornate. Con le tre retrocesse che salutano la massima serie, il campionato si prepara ora alla nuova edizione 2025/2026, già iniziata con nuovi equilibri e nuove sfide. Il campionato italiano di massima serie vede la partecipazione di 20 squadre che si affrontano in un girone unico con formula di andata e ritorno, per un totale di 38 giornate. Sì, oltre alla sospensione di Fiorentina-Inter per il malore di un giocatore, si segnalano i rinvii della 33ª giornata per il

proceed? y


{'messages': [ToolMessage(content='[{\'title\': \'Monte ingaggi Napoli 2025/26: tutti gli stipendi aggiornati degli azzurri | Transfermarkt\', \'content\': "Kevin De Bruyne resta il giocatore più pagato della rosa con 11,11 milioni di euro lordi, davanti a Rasmus Højlund e Romelu Lukaku. Gli stipendi più alti della Serie A  La Top11 per stipendi del Napoli 2025/26 totalizza 80 milioni di euro lordi. Ieri 10:51 Serie A Media spettatori: Inter e Milan guadagnano un posto, ma chi è davanti? Ieri 10:51 Transfermarkt Media spettatori: Inter e Milan guadagnano un posto, ma chi è davanti? 13/04/2026 14:45 Transfermarkt Serie A e giovani: Como, valori alle stelle ed esempio per le big. 10/04/2026 14:45 Transfermarkt Calendario: ecco il (lieve) vantaggio del Napoli. 08/02/2026 07:45 Transfermarkt Stipendi Top in Serie A: il rinnovo di Yildiz sposta gli equilibri? 01/12/2025 07:46 Transfermarkt Quanto vale il Napoli? 24/03/2026 13:00 Transfermarkt Serie A coi nuovi valori: crolla l\'Inter ma sal

## Modifichiamo ora lo stato tra un'esecuzione e l'altra

In [90]:
thread = {
    "configurable": {
        "thread_id": "asdasdasd"
    }
}
for event in agent.graph.stream({"messages": [HumanMessage('Chi ha vinto il campionato di calcio serie A nel 2025? Qual è il giocatore più pagato quell anno di quella squadra?')]}, thread):
    for v in event.values():
        print(v)

agent.graph.get_state(thread)

{'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"query": "chi ha vinto campionato serie A 2025"}'}, '__gemini_function_call_thought_signatures__': {'a9852a24-7624-487a-bde3-db6c011a57e8': 'CsoDAQw51schlzK5AMAuzdURrxjdskVx50PG3O5s8pa+XMqmwVfvAHYhDnBwekzwlUMiERS0oQKpUABf/fojhQecdak7dzSXpNFq1c7wtxOoafi1tcyff+yQc448DXF0RcMe3+zeRuXse6pswSR8gvtjUhypuuBh7xjLiZ7Bf2q1VCBSt8sUX5t352h5a7HstGg4Y+jmYsBRrIvH9eteLdYI/x84xnB3F0c5qQXpHS/FGgFyzgcMWNb7kzgXrqs9PWEfeGZWHD7y+k0fbJ0qAkcIlyEpcM1ui9NsqA6JomnShVqYqISo+UifpOP1gLE8yEWQjWSkvQ9xf8YTW2PyRIn+8hyKuqZcJNsRMccLBKNpF6P72BYqXsswGku6zfoSWxAmbg0232rCCCaPyiudTr5rVNGV/LLjV3021NtO+Ywcxm+i5gDNJpn+5uMYp09vFT4l2nYmFtaInP7ryx9wcgqkTIdAlz59N8BsX9vGelc9KloqyR+FozDecMDrkH1I7I+J+ZzS3U67Mz9EH93IYWSKvVbHENALpESOyH1TsSc38moxrXrqUjjPF9BPMqzfJR0u4C74vWF99igG8gyXhuJ6MkVwoSjF4HLYygM='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'g

StateSnapshot(values={'messages': [HumanMessage(content='Chi ha vinto il campionato di calcio serie A nel 2025? Qual è il giocatore più pagato quell anno di quella squadra?', additional_kwargs={}, response_metadata={}, id='e34362ec-8ef1-48b9-a151-26db69d8a4ea'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"query": "chi ha vinto campionato serie A 2025"}'}, '__gemini_function_call_thought_signatures__': {'a9852a24-7624-487a-bde3-db6c011a57e8': 'CsoDAQw51schlzK5AMAuzdURrxjdskVx50PG3O5s8pa+XMqmwVfvAHYhDnBwekzwlUMiERS0oQKpUABf/fojhQecdak7dzSXpNFq1c7wtxOoafi1tcyff+yQc448DXF0RcMe3+zeRuXse6pswSR8gvtjUhypuuBh7xjLiZ7Bf2q1VCBSt8sUX5t352h5a7HstGg4Y+jmYsBRrIvH9eteLdYI/x84xnB3F0c5qQXpHS/FGgFyzgcMWNb7kzgXrqs9PWEfeGZWHD7y+k0fbJ0qAkcIlyEpcM1ui9NsqA6JomnShVqYqISo+UifpOP1gLE8yEWQjWSkvQ9xf8YTW2PyRIn+8hyKuqZcJNsRMccLBKNpF6P72BYqXsswGku6zfoSWxAmbg0232rCCCaPyiudTr5rVNGV/LLjV3021NtO+Ywcxm+i5gDNJpn+5uMYp09vFT4l2nYmFtaInP7ryx9wcgqkTIdAlz59N8BsX9vGelc9

In [91]:
current_values = agent.graph.get_state(thread)

In [92]:
current_values.values['messages'][-1]

AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"query": "chi ha vinto campionato serie A 2025"}'}, '__gemini_function_call_thought_signatures__': {'a9852a24-7624-487a-bde3-db6c011a57e8': 'CsoDAQw51schlzK5AMAuzdURrxjdskVx50PG3O5s8pa+XMqmwVfvAHYhDnBwekzwlUMiERS0oQKpUABf/fojhQecdak7dzSXpNFq1c7wtxOoafi1tcyff+yQc448DXF0RcMe3+zeRuXse6pswSR8gvtjUhypuuBh7xjLiZ7Bf2q1VCBSt8sUX5t352h5a7HstGg4Y+jmYsBRrIvH9eteLdYI/x84xnB3F0c5qQXpHS/FGgFyzgcMWNb7kzgXrqs9PWEfeGZWHD7y+k0fbJ0qAkcIlyEpcM1ui9NsqA6JomnShVqYqISo+UifpOP1gLE8yEWQjWSkvQ9xf8YTW2PyRIn+8hyKuqZcJNsRMccLBKNpF6P72BYqXsswGku6zfoSWxAmbg0232rCCCaPyiudTr5rVNGV/LLjV3021NtO+Ywcxm+i5gDNJpn+5uMYp09vFT4l2nYmFtaInP7ryx9wcgqkTIdAlz59N8BsX9vGelc9KloqyR+FozDecMDrkH1I7I+J+ZzS3U67Mz9EH93IYWSKvVbHENALpESOyH1TsSc38moxrXrqUjjPF9BPMqzfJR0u4C74vWF99igG8gyXhuJ6MkVwoSjF4HLYygM='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'},

In [93]:
current_values.values['messages'][-1].tool_calls

[{'name': 'tavily_search_tool',
  'args': {'query': 'chi ha vinto campionato serie A 2025'},
  'id': 'a9852a24-7624-487a-bde3-db6c011a57e8',
  'type': 'tool_call'}]

In [94]:
#Cambio la tool call
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search_tool',
  'args': {'query': 'chi ha vinto il campionato di calcio serie A nel 2022'},
  'id': _id}
]

agent.graph.update_state(thread, current_values.values)

{'configurable': {'thread_id': 'asdasdasd',
  'checkpoint_ns': '',
  'checkpoint_id': '1f139be9-2aea-62dd-8002-3f150b500c2a'}}

In [95]:
#ADesso chiamerà Tavily search con la query cambiata
for event in agent.graph.stream(None, thread):
    for v in event.values():
        print(v)

{'messages': [ToolMessage(content='[{\'title\': \'Serie A calcio 2022: classifica finale e verdetti. Scudetto, qualificate ...\', \'content\': "Il Milan si è laureato Campione d\'Italia, i rossoneri hanno conquistato il 19mo scudetto della storia. Il Diavolo è tornato a vincere dopo", \'url\': \'https://www.oasport.it/2022/05/serie-a-calcio-2022-classifica-finale-e-verdetti-scudetto-qualificate-a-champions-league-ed-europa-league-retrocesse-in-serie-b/\'}, {\'title\': \'2021–22 Serie A - Wikipedia\', \'content\': \'AC Milan won the title after defeating Sassuolo 3–0 at the Mapei Stadium – Città del Tricolore on 22 May 2022, the final matchday of the season. Serie A. Season\', \'url\': \'https://en.wikipedia.org/wiki/2021%E2%80%9322_Serie_A\'}, {\'title\': \'Classifica Serie A 2022/2023 - la Repubblica\', \'content\': \'## calcio. # Classifica Serie A 2022/2023. | Squadra | Pt | PG | V | N | P | GF | GS | Gruppo |. | Napoli | 90 | 38 | 28 | 6 | 4 | 77 | 28 |  |. | Lazio | 74 | 38 | 22 |

In [96]:
#Possiamo anche recuperare gli stati precedenti
states = []
for state in agent.graph.get_state_history(thread):
    print(state)
    print('--')
    states.append(state)

StateSnapshot(values={'messages': [{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'schema', 'messages', 'HumanMessage'], 'kwargs': {'content': 'Chi ha vinto il campionato di calcio serie A nel 2025? Qual è il giocatore più pagato quell anno di quella squadra?', 'type': 'human', 'id': 'e34362ec-8ef1-48b9-a151-26db69d8a4ea'}}, {'lc': 1, 'type': 'constructor', 'id': ['langchain', 'schema', 'messages', 'AIMessage'], 'kwargs': {'content': '', 'additional_kwargs': {'function_call': {'name': 'tavily_search_tool', 'arguments': '{"query": "chi ha vinto campionato serie A 2025"}'}, '__gemini_function_call_thought_signatures__': {'a9852a24-7624-487a-bde3-db6c011a57e8': 'CsoDAQw51schlzK5AMAuzdURrxjdskVx50PG3O5s8pa+XMqmwVfvAHYhDnBwekzwlUMiERS0oQKpUABf/fojhQecdak7dzSXpNFq1c7wtxOoafi1tcyff+yQc448DXF0RcMe3+zeRuXse6pswSR8gvtjUhypuuBh7xjLiZ7Bf2q1VCBSt8sUX5t352h5a7HstGg4Y+jmYsBRrIvH9eteLdYI/x84xnB3F0c5qQXpHS/FGgFyzgcMWNb7kzgXrqs9PWEfeGZWHD7y+k0fbJ0qAkcIlyEpcM1ui9NsqA6JomnShVqYqISo+UifpOP1gLE8yEWQjW

In [114]:
to_replay = states[-3]
to_replay
#Siamo tornati a quando l'agent chiamava tavily per cercare chi ha vinto il campionato di serie A 2025
from langchain_core.load import load

In [115]:
for event in agent.graph.stream(None, to_replay.config):
    for k, v in event.items():
        print(v)

{'messages': [ToolMessage(content="[{'title': 'Calcio Serie A - Classifica Finale 2024/2025 Napoli Campione - Italiavista.it', 'content': 'Il campionato, 123ª edizione della massima serie italiana e 93ª a girone unico, ha visto l’Inter chiudere al secondo posto con un solo punto di distacco, mentre l’Atalanta ha conquistato il podio sorprendendo molti pronostici iniziali. La Serie A rappresenta il vertice del calcio professionistico italiano, organizzata in regime di girone unico con partite di andata e ritorno per un totale di 38 giornate. Con le tre retrocesse che salutano la massima serie, il campionato si prepara ora alla nuova edizione 2025/2026, già iniziata con nuovi equilibri e nuove sfide. Il campionato italiano di massima serie vede la partecipazione di 20 squadre che si affrontano in un girone unico con formula di andata e ritorno, per un totale di 38 giornate. Sì, oltre alla sospensione di Fiorentina-Inter per il malore di un giocatore, si segnalano i rinvii della 33ª giorn

In [121]:
#ORA CAMBIAMO LA TOOL CALL
to_replay_obj = load(to_replay.values)
_id = to_replay_obj['messages'][-1].tool_calls[0]['id']

to_replay_obj['messages'][-1].tool_calls = [{'name': 'tavily_search_tool',
  'args': {'query': 'vincitore serie A '},
  'id': _id}]

{'messages': [HumanMessage(content='Chi ha vinto il campionato di calcio serie A nel 2025? Qual è il giocatore più pagato quell anno di quella squadra?', additional_kwargs={}, response_metadata={}, id='e34362ec-8ef1-48b9-a151-26db69d8a4ea'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"query": "chi ha vinto campionato serie A 2025"}'}, '__gemini_function_call_thought_signatures__': {'a9852a24-7624-487a-bde3-db6c011a57e8': 'CsoDAQw51schlzK5AMAuzdURrxjdskVx50PG3O5s8pa+XMqmwVfvAHYhDnBwekzwlUMiERS0oQKpUABf/fojhQecdak7dzSXpNFq1c7wtxOoafi1tcyff+yQc448DXF0RcMe3+zeRuXse6pswSR8gvtjUhypuuBh7xjLiZ7Bf2q1VCBSt8sUX5t352h5a7HstGg4Y+jmYsBRrIvH9eteLdYI/x84xnB3F0c5qQXpHS/FGgFyzgcMWNb7kzgXrqs9PWEfeGZWHD7y+k0fbJ0qAkcIlyEpcM1ui9NsqA6JomnShVqYqISo+UifpOP1gLE8yEWQjWSkvQ9xf8YTW2PyRIn+8hyKuqZcJNsRMccLBKNpF6P72BYqXsswGku6zfoSWxAmbg0232rCCCaPyiudTr5rVNGV/LLjV3021NtO+Ywcxm+i5gDNJpn+5uMYp09vFT4l2nYmFtaInP7ryx9wcgqkTIdAlz59N8BsX9vGelc9KloqyR+FozDecMDrkH1I7

In [124]:
branch_state = agent.graph.update_state(to_replay.config, to_replay_obj)

In [126]:
for event in agent.graph.stream(None, branch_state):
    for k, v in event.items():
        if k != "__end__":
            print(v)

{'messages': [ToolMessage(content='[{\'title\': "Serie A, l\'albo d\'oro dei vincitori e la lista completa degli scudetti · Calcio", \'content\': "La stagione 2024 / 25 di **Serie A** si è conclusa con la vittoria dello Scudetto da parte del **Napoli**. Si tratta del 4° Scudetto per la squadra partenopea, guidata da Antonio Conte, che in carriera è stato Campione d\'Italia sulle panchine di Juventus e Inter, prima di completare uno storico tris col Napoli. La Juventus è il club italiano di calcio che vanta il maggior numero di vittorie del campionato di Serie A con 36 scudetti. To reject cookies or manage your cookies preferences click No, manage settings. Please note that blocking some types of cookies may impact your experience of the Digital Platforms and the services we are able to offer. Strictly Necessary cookies that enable you to navigate our Digital Platforms and enable us to provide an optimised service.", \'url\': \'https://www.olympics.com/it/notizie/calcio-serie-a-elenco-c